# Module 8 • Large Language Models

# Lesson 49 • Multimodal Large Language Models and Vision-Language Foundations

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Advanced  
**Estimated study time:** 180–220 minutes  
**Execution target:** CPU only

---

## Scope

This lesson introduces multimodal large language models (MLLMs) and
vision-language modeling.

The executable core is fully offline and demonstrates:

- synthetic image construction;
- simple visual feature extraction;
- text feature extraction;
- projection into a shared embedding space;
- contrastive image-text alignment;
- image-text retrieval;
- visual question answering;
- image captioning concepts;
- multimodal prompting;
- modality fusion;
- retrieval and VQA evaluation;
- robustness under visual perturbations;
- multilingual and Arabic multimodal considerations.

The notebook does not download a vision-language model. The local experiments
illustrate the architecture and evaluation mechanics in a CPU-safe way.

## Learning Objectives

After completing this lesson, the learner should be able to:

- define multimodal language modeling;
- explain how a vision encoder differs from a language model;
- describe projection layers between visual and textual representations;
- explain contrastive image-text alignment;
- compute image-text similarities;
- perform cross-modal retrieval;
- distinguish early, late, and intermediate fusion;
- explain visual question answering;
- explain multimodal caption generation;
- design multimodal prompts;
- evaluate image-text retrieval and VQA;
- identify hallucination risks in multimodal models;
- discuss multilingual and Arabic vision-language modeling.

## Table of Contents

1. What Is Multimodal AI?
2. Modalities
3. Vision Encoders
4. Language Models
5. Projection Layers
6. Shared Embedding Spaces
7. Contrastive Learning
8. Image-Text Retrieval
9. Vision-Language Fusion
10. Early, Intermediate, and Late Fusion
11. Visual Question Answering
12. Image Captioning
13. Multimodal Prompting
14. Region-Level Reasoning
15. OCR and Text in Images
16. Spatial Reasoning
17. Multimodal Hallucination
18. Offline Synthetic Image Dataset
19. Visual Feature Extraction
20. Text Feature Extraction
21. Projection into Shared Space
22. Contrastive Alignment
23. Image-to-Text Retrieval
24. Text-to-Image Retrieval
25. Retrieval Metrics
26. Recall@k
27. Mean Reciprocal Rank
28. Synthetic VQA Dataset
29. Deterministic VQA Baseline
30. VQA Accuracy
31. Caption Generation Baseline
32. Caption Evaluation
33. Multimodal Prompt Templates
34. Visual Perturbation Robustness
35. Modality Ablation
36. Fusion Experiment
37. Failure Taxonomy
38. OCR Failure Modes
39. Spatial Failure Modes
40. Safety and Multimodal Inputs
41. Multilingual Multimodality
42. Arabic Multimodal NLP
43. Optional Hugging Face Workflow
44. Reproducibility
45. Knowledge Check
46. Exercises
47. Summary and Next Lesson

# 1. What Is Multimodal AI?

Multimodal AI processes more than one type of signal, such as:

- text;
- images;
- audio;
- video;
- sensor streams.

A multimodal large language model usually combines a language model with one or
more encoders for non-text modalities.

In [ ]:
import importlib.util
import math
import platform
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

modalities = pd.DataFrame(
    [
        ("Text", "tokens"),
        ("Image", "pixels or patches"),
        ("Audio", "waveform or spectrogram"),
        ("Video", "frames over time"),
        ("Sensors", "numeric time series"),
    ],
    columns=["Modality", "Typical representation"],
)

modalities

# 2. Modalities

Each modality has its own structure. Images are spatial, audio is temporal, and
text is sequential.

Multimodal systems need mechanisms to convert these signals into compatible
representations.

# 3. Vision Encoders

Vision encoders can include:

- convolutional neural networks;
- Vision Transformers;
- hierarchical vision transformers.

Their output is often a set of visual token or patch representations.

# 4. Language Models

The language model receives textual tokens and, in MLLMs, may also receive
transformed visual representations.

# 5. Projection Layers

A projection layer maps vision features into a dimension compatible with the
language-model hidden space.

Conceptually:

\[
z_{vision} = W h_{vision}
\]

where \(h_{vision}\) is the visual representation and \(W\) is a learned
projection.

# 6. Shared Embedding Spaces

Image-text models can map images and captions into a common vector space.

Matching image-text pairs should have high similarity; unrelated pairs should have
lower similarity.

# 7. Contrastive Learning

Contrastive learning trains matched image-text pairs to be close while pushing
mismatched pairs apart.

A typical batch contains:

```text
image_1 <-> caption_1
image_2 <-> caption_2
...
```

# 8. Image-Text Retrieval

Once images and text are embedded in a shared space, we can retrieve:

- captions for an image;
- images for a text query.

# 9. Vision-Language Fusion

Fusion determines how visual and textual information interact.

# 10. Early, Intermediate, and Late Fusion

In [ ]:
fusion_table = pd.DataFrame(
    [
        ("Early", "combine low-level features early", "high interaction, high cost"),
        ("Intermediate", "fuse learned modality representations", "common practical design"),
        ("Late", "combine final predictions or scores", "simple but less expressive"),
    ],
    columns=["Fusion type", "Mechanism", "Trade-off"],
)

fusion_table

# 11. Visual Question Answering

Visual question answering (VQA) requires both:

- understanding the image;
- understanding the question.

Example:

```text
Image: red square
Question: What color is the square?
Answer: red
```

# 12. Image Captioning

Image captioning generates a textual description of visual content.

Evaluation should distinguish:

- object correctness;
- attribute correctness;
- relation correctness;
- fluency.

# 13. Multimodal Prompting

A multimodal prompt may combine:

- system instruction;
- image;
- user question;
- output format constraint.

# 14. Region-Level Reasoning

Region-level tasks include:

- referring expression resolution;
- object localization;
- region captioning;
- counting.

# 15. OCR and Text in Images

Vision-language systems often need to recognize embedded text.

OCR errors can propagate into downstream reasoning.

# 16. Spatial Reasoning

Spatial relations include:

- left/right;
- above/below;
- inside/outside;
- near/far;
- relative size.

# 17. Multimodal Hallucination

A multimodal model can hallucinate:

- nonexistent objects;
- incorrect colors;
- wrong counts;
- unsupported relationships;
- text that is not present in the image.

# 18. Offline Synthetic Image Dataset

We create small synthetic images containing colored geometric shapes.

In [ ]:
IMAGE_SIZE = 32

COLOR_VALUES = {
    "red": np.array([1.0, 0.0, 0.0]),
    "green": np.array([0.0, 1.0, 0.0]),
    "blue": np.array([0.0, 0.0, 1.0]),
}


def create_shape_image(
    color: str,
    shape: str,
    position: str,
) -> np.ndarray:
    image = np.zeros(
        (
            IMAGE_SIZE,
            IMAGE_SIZE,
            3,
        ),
        dtype=float,
    )

    center_x = {
        "left": 9,
        "center": 16,
        "right": 23,
    }[position]

    center_y = 16

    yy, xx = np.mgrid[
        0:IMAGE_SIZE,
        0:IMAGE_SIZE,
    ]

    if shape == "square":
        mask = (
            (np.abs(xx - center_x) <= 5)
            & (np.abs(yy - center_y) <= 5)
        )
    elif shape == "circle":
        mask = (
            (xx - center_x) ** 2
            + (yy - center_y) ** 2
            <= 5 ** 2
        )
    else:
        raise ValueError(
            "Unsupported shape."
        )

    image[
        mask
    ] = COLOR_VALUES[
        color
    ]

    return image


synthetic_records = []

for color in COLOR_VALUES:
    for shape in [
        "square",
        "circle",
    ]:
        for position in [
            "left",
            "center",
            "right",
        ]:
            synthetic_records.append(
                {
                    "color": color,
                    "shape": shape,
                    "position": position,
                    "caption": (
                        f"a {color} {shape} on the {position}"
                    ),
                    "image": create_shape_image(
                        color,
                        shape,
                        position,
                    ),
                }
            )

len(synthetic_records)

In [ ]:
example_image = synthetic_records[
    0
]["image"]

plt.figure(figsize=(4, 4))
plt.imshow(example_image)
plt.axis("off")
plt.title(
    synthetic_records[
        0
    ]["caption"]
)
plt.tight_layout()
plt.show()

# 19. Visual Feature Extraction

We extract interpretable visual features:

- mean RGB values;
- occupied-pixel centroid;
- occupied area;
- compactness proxy.

A real model would learn features automatically.

In [ ]:
def visual_features(
    image: np.ndarray,
) -> np.ndarray:
    occupied = (
        image.sum(
            axis=2
        )
        > 0
    )

    mean_rgb = image[
        occupied
    ].mean(
        axis=0
    )

    yy, xx = np.where(
        occupied
    )

    centroid_x = (
        xx.mean()
        / IMAGE_SIZE
    )

    centroid_y = (
        yy.mean()
        / IMAGE_SIZE
    )

    area = (
        occupied.mean()
    )

    width = (
        xx.max()
        - xx.min()
        + 1
    )

    height = (
        yy.max()
        - yy.min()
        + 1
    )

    bounding_area = (
        width
        * height
        / (
            IMAGE_SIZE
            * IMAGE_SIZE
        )
    )

    compactness = (
        area
        / max(
            bounding_area,
            1e-8,
        )
    )

    return np.array(
        [
            *mean_rgb,
            centroid_x,
            centroid_y,
            area,
            compactness,
        ],
        dtype=float,
    )


visual_features(
    example_image
)

# 20. Text Feature Extraction

The text encoder uses known attributes from captions to build a small semantic
feature vector.

In [ ]:
TEXT_FEATURE_NAMES = [
    "red",
    "green",
    "blue",
    "square",
    "circle",
    "left",
    "center",
    "right",
]


def text_features(
    text: str,
) -> np.ndarray:
    lowered = text.lower()

    return np.array(
        [
            float(
                feature in lowered
            )
            for feature
            in TEXT_FEATURE_NAMES
        ],
        dtype=float,
    )


text_features(
    "a red square on the left"
)

# 21. Projection into Shared Space

We learn linear projections from visual and textual features into a common space
using paired examples.

In [ ]:
visual_matrix = np.vstack(
    [
        visual_features(
            record["image"]
        )
        for record
        in synthetic_records
    ]
)

text_matrix = np.vstack(
    [
        text_features(
            record["caption"]
        )
        for record
        in synthetic_records
    ]
)

visual_targets = text_matrix.copy()

visual_projection, *_ = (
    np.linalg.lstsq(
        visual_matrix,
        visual_targets,
        rcond=None,
    )
)

projected_visual = (
    visual_matrix
    @ visual_projection
)

projected_text = (
    text_matrix.copy()
)

print(
    "Projected visual shape:",
    projected_visual.shape,
)

# 22. Contrastive Alignment

We normalize embeddings and inspect matched versus mismatched cosine similarities.

In [ ]:
def row_normalize(
    matrix: np.ndarray,
) -> np.ndarray:
    norms = np.linalg.norm(
        matrix,
        axis=1,
        keepdims=True,
    )

    return (
        matrix
        / np.maximum(
            norms,
            1e-12,
        )
    )


normalized_visual = row_normalize(
    projected_visual
)

normalized_text = row_normalize(
    projected_text
)

similarity_matrix = (
    normalized_visual
    @ normalized_text.T
)

matched_scores = np.diag(
    similarity_matrix
)

mismatched_scores = similarity_matrix[
    ~np.eye(
        len(similarity_matrix),
        dtype=bool,
    )
]

pd.Series(
    {
        "mean matched similarity": (
            matched_scores.mean()
        ),
        "mean mismatched similarity": (
            mismatched_scores.mean()
        ),
    }
)

# 23. Image-to-Text Retrieval

In [ ]:
def image_to_text_retrieval(
    image_index: int,
    top_k: int = 3,
) -> pd.DataFrame:
    scores = similarity_matrix[
        image_index
    ]

    indices = np.argsort(
        scores
    )[::-1][
        :top_k
    ]

    return pd.DataFrame(
        [
            {
                "rank": rank,
                "score": float(
                    scores[index]
                ),
                "caption": synthetic_records[
                    int(index)
                ]["caption"],
            }
            for rank, index
            in enumerate(
                indices,
                start=1,
            )
        ]
    )


image_to_text_retrieval(
    0,
    top_k=5,
)

# 24. Text-to-Image Retrieval

In [ ]:
def text_to_image_retrieval(
    text_index: int,
    top_k: int = 3,
) -> pd.DataFrame:
    scores = similarity_matrix[
        :,
        text_index
    ]

    indices = np.argsort(
        scores
    )[::-1][
        :top_k
    ]

    return pd.DataFrame(
        [
            {
                "rank": rank,
                "score": float(
                    scores[index]
                ),
                "image_caption": synthetic_records[
                    int(index)
                ]["caption"],
            }
            for rank, index
            in enumerate(
                indices,
                start=1,
            )
        ]
    )


text_to_image_retrieval(
    4,
    top_k=5,
)

# 25. Retrieval Metrics

Multimodal retrieval commonly reports recall@k and ranking metrics.

# 26. Recall@k

In [ ]:
def retrieval_recall_at_k(
    similarity_matrix: np.ndarray,
    k: int,
) -> float:
    successes = []

    for row_index in range(
        similarity_matrix.shape[0]
    ):
        top_indices = np.argsort(
            similarity_matrix[
                row_index
            ]
        )[::-1][
            :k
        ]

        successes.append(
            row_index
            in top_indices
        )

    return float(
        np.mean(
            successes
        )
    )


pd.Series(
    {
        "R@1": (
            retrieval_recall_at_k(
                similarity_matrix,
                1,
            )
        ),
        "R@3": (
            retrieval_recall_at_k(
                similarity_matrix,
                3,
            )
        ),
        "R@5": (
            retrieval_recall_at_k(
                similarity_matrix,
                5,
            )
        ),
    }
)

# 27. Mean Reciprocal Rank

In [ ]:
def retrieval_mrr(
    similarity_matrix: np.ndarray,
) -> float:
    reciprocal_ranks = []

    for row_index in range(
        similarity_matrix.shape[0]
    ):
        ranking = np.argsort(
            similarity_matrix[
                row_index
            ]
        )[::-1]

        rank = int(
            np.where(
                ranking
                == row_index
            )[0][0]
        ) + 1

        reciprocal_ranks.append(
            1.0 / rank
        )

    return float(
        np.mean(
            reciprocal_ranks
        )
    )


retrieval_mrr(
    similarity_matrix
)

# 28. Synthetic VQA Dataset

In [ ]:
vqa_examples = []

for index, record in enumerate(
    synthetic_records
):
    vqa_examples.extend(
        [
            {
                "image_index": index,
                "question": (
                    "What color is the shape?"
                ),
                "answer": (
                    record["color"]
                ),
                "type": "color",
            },
            {
                "image_index": index,
                "question": (
                    "What shape is shown?"
                ),
                "answer": (
                    record["shape"]
                ),
                "type": "shape",
            },
            {
                "image_index": index,
                "question": (
                    "Where is the shape?"
                ),
                "answer": (
                    record["position"]
                ),
                "type": "position",
            },
        ]
    )

len(vqa_examples)

# 29. Deterministic VQA Baseline

We recover visual attributes from image features.

In [ ]:
def infer_color(
    image: np.ndarray,
) -> str:
    occupied = (
        image.sum(
            axis=2
        )
        > 0
    )

    mean_rgb = image[
        occupied
    ].mean(
        axis=0
    )

    return [
        "red",
        "green",
        "blue",
    ][
        int(
            np.argmax(
                mean_rgb
            )
        )
    ]


def infer_position(
    image: np.ndarray,
) -> str:
    occupied = (
        image.sum(
            axis=2
        )
        > 0
    )

    _, xx = np.where(
        occupied
    )

    center = xx.mean()

    if center < 12:
        return "left"

    if center > 20:
        return "right"

    return "center"


def infer_shape(
    image: np.ndarray,
) -> str:
    features = visual_features(
        image
    )

    compactness = features[
        -1
    ]

    if compactness > 0.90:
        return "square"

    return "circle"


def answer_visual_question(
    image: np.ndarray,
    question: str,
) -> str:
    lowered = question.lower()

    if "color" in lowered:
        return infer_color(
            image
        )

    if "shape" in lowered:
        return infer_shape(
            image
        )

    if (
        "where" in lowered
        or "position" in lowered
    ):
        return infer_position(
            image
        )

    return "unknown"


answer_visual_question(
    synthetic_records[
        0
    ]["image"],
    "What color is the shape?",
)

# 30. VQA Accuracy

In [ ]:
vqa_rows = []

for example in vqa_examples:
    prediction = (
        answer_visual_question(
            synthetic_records[
                example[
                    "image_index"
                ]
            ]["image"],
            example[
                "question"
            ],
        )
    )

    vqa_rows.append(
        {
            **example,
            "prediction": (
                prediction
            ),
            "correct": (
                prediction
                == example[
                    "answer"
                ]
            ),
        }
    )

vqa_frame = pd.DataFrame(
    vqa_rows
)

vqa_frame.groupby(
    "type"
)["correct"].mean()

# 31. Caption Generation Baseline

In [ ]:
def generate_caption(
    image: np.ndarray,
) -> str:
    return (
        f"a {infer_color(image)} "
        f"{infer_shape(image)} "
        f"on the {infer_position(image)}"
    )


generated_captions = [
    generate_caption(
        record["image"]
    )
    for record
    in synthetic_records
]

generated_captions[:5]

# 32. Caption Evaluation

In [ ]:
def token_f1(
    reference: str,
    prediction: str,
) -> float:
    reference_tokens = (
        reference.lower().split()
    )
    prediction_tokens = (
        prediction.lower().split()
    )

    overlap = sum(
        (
            Counter(
                reference_tokens
            )
            & Counter(
                prediction_tokens
            )
        ).values()
    )

    if overlap == 0:
        return 0.0

    precision = (
        overlap
        / len(
            prediction_tokens
        )
    )

    recall = (
        overlap
        / len(
            reference_tokens
        )
    )

    return (
        2
        * precision
        * recall
        / (
            precision
            + recall
        )
    )


caption_scores = [
    token_f1(
        record["caption"],
        prediction,
    )
    for record, prediction
    in zip(
        synthetic_records,
        generated_captions,
    )
]

pd.Series(
    {
        "mean caption token F1": (
            np.mean(
                caption_scores
            )
        )
    }
)

# 33. Multimodal Prompt Templates

In [ ]:
MULTIMODAL_PROMPT = (
    "You are a vision-language assistant.\n\n"
    "Task:\n"
    "Answer the question using only information visible in the image.\n\n"
    "Question:\n"
    "{question}\n\n"
    "Return a concise answer.\n"
)


def build_multimodal_prompt(
    question: str,
) -> str:
    return MULTIMODAL_PROMPT.format(
        question=question
    )


print(
    build_multimodal_prompt(
        "What color is the shape?"
    )
)


# 34. Visual Perturbation Robustness

We test robustness under small pixel noise.

In [ ]:
rng = np.random.default_rng(
    42
)


def add_visual_noise(
    image: np.ndarray,
    noise_std: float,
) -> np.ndarray:
    noisy = (
        image
        + rng.normal(
            0.0,
            noise_std,
            size=image.shape,
        )
    )

    return np.clip(
        noisy,
        0.0,
        1.0,
    )


robustness_rows = []

for noise_std in [
    0.0,
    0.02,
    0.05,
    0.10,
]:
    correct = []

    for record in synthetic_records:
        noisy_image = (
            add_visual_noise(
                record["image"],
                noise_std,
            )
        )

        correct.append(
            generate_caption(
                noisy_image
            )
            == record[
                "caption"
            ]
        )

    robustness_rows.append(
        {
            "noise_std": noise_std,
            "caption_exact_accuracy": (
                np.mean(
                    correct
                )
            ),
        }
    )

robustness_frame = pd.DataFrame(
    robustness_rows
)

robustness_frame

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(
    robustness_frame[
        "noise_std"
    ],
    robustness_frame[
        "caption_exact_accuracy"
    ],
    marker="o",
)
plt.xlabel("Noise standard deviation")
plt.ylabel("Exact caption accuracy")
plt.title("Visual Perturbation Robustness")
plt.tight_layout()
plt.show()

# 35. Modality Ablation

Ablation asks how performance changes when one modality is removed.

In [ ]:
modality_ablation = pd.DataFrame(
    [
        (
            "Image + question",
            "full VQA",
        ),
        (
            "Question only",
            "language prior only",
        ),
        (
            "Image only",
            "cannot identify requested task reliably",
        ),
    ],
    columns=[
        "Input",
        "Expected capability",
    ],
)

modality_ablation

# 36. Fusion Experiment

We combine visual and text attribute vectors to show intermediate fusion.

In [ ]:
def fused_representation(
    image: np.ndarray,
    question: str,
) -> np.ndarray:
    image_vector = (
        visual_features(
            image
        )
    )

    question_features = np.array(
        [
            float(
                "color"
                in question.lower()
            ),
            float(
                "shape"
                in question.lower()
            ),
            float(
                (
                    "where"
                    in question.lower()
                )
                or (
                    "position"
                    in question.lower()
                )
            ),
        ]
    )

    return np.concatenate(
        [
            image_vector,
            question_features,
        ]
    )


fused_representation(
    synthetic_records[
        0
    ]["image"],
    "What color is the shape?",
)

# 37. Failure Taxonomy

In [ ]:
multimodal_failures = pd.DataFrame(
    [
        ("Object hallucination", "mentions object not present"),
        ("Attribute error", "wrong color or property"),
        ("Counting error", "wrong number of objects"),
        ("Spatial error", "wrong relation or position"),
        ("OCR error", "misreads text in image"),
        ("Question misunderstanding", "visual evidence correct but task misread"),
        ("Cross-modal mismatch", "wrong image-text association"),
    ],
    columns=[
        "Failure",
        "Description",
    ],
)

multimodal_failures

# 38. OCR Failure Modes

OCR failures can result from:

- small text;
- low contrast;
- stylized fonts;
- rotation;
- Arabic ligatures;
- mixed scripts.

# 39. Spatial Failure Modes

Spatial reasoning may fail when:

- objects overlap;
- the viewpoint changes;
- coordinates are implicit;
- relative position is ambiguous.

# 40. Safety and Multimodal Inputs

Images can contain:

- personal data;
- misleading text;
- instruction-like content;
- unsafe visual material.

Multimodal systems therefore require input-safety and privacy controls.

# 41. Multilingual Multimodality

Multilingual multimodal systems must align:

- visual concepts;
- multiple languages;
- multiple scripts;
- language-specific caption distributions.

# 42. Arabic Multimodal NLP

Arabic vision-language tasks include:

- Arabic image captioning;
- Arabic VQA;
- OCR in Arabic images;
- document understanding;
- scene-text understanding.

In [ ]:
arabic_multimodal_examples = pd.DataFrame(
    [
        (
            "مَا لَوْنُ الشَّكْلِ؟",
            "What color is the shape?",
            "fully vocalized VQA question",
        ),
        (
            "صِفِ الصُّورَةَ فِي جُمْلَةٍ وَاحِدَةٍ.",
            "Describe the image in one sentence.",
            "captioning instruction",
        ),
        (
            "اِقْرَأِ النَّصَّ الظَّاهِرَ فِي الصُّورَةِ.",
            "Read the text visible in the image.",
            "Arabic OCR instruction",
        ),
    ],
    columns=[
        "Arabic prompt",
        "English meaning",
        "Task",
    ],
)

arabic_multimodal_examples

For fully vocalized Arabic multimodal tasks, tashkeel preservation should be
evaluated explicitly in captions, OCR output, VQA answers, and generated text.

# 43. Optional Hugging Face Workflow

The following template is disabled by default.

In [ ]:
TRANSFORMERS_AVAILABLE = (
    importlib.util.find_spec(
        "transformers"
    )
    is not None
)

RUN_MULTIMODAL_DEMO = False

multimodal_template = '''
from transformers import AutoProcessor, AutoModelForVision2Seq

processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID
)

inputs = processor(
    images=image,
    text=prompt,
    return_tensors="pt",
)

generated = model.generate(
    **inputs,
    max_new_tokens=64,
)

answer = processor.batch_decode(
    generated,
    skip_special_tokens=True,
)
'''

print(
    "transformers installed:",
    TRANSFORMERS_AVAILABLE,
)
print(
    "Optional multimodal demo enabled:",
    RUN_MULTIMODAL_DEMO,
)
print(
    multimodal_template
)

# 44. Reproducibility

Record:

- vision encoder;
- language model;
- projection layer;
- image preprocessing;
- image resolution;
- text tokenizer;
- prompt template;
- fusion method;
- decoding settings;
- evaluation dataset;
- retrieval metrics;
- VQA metrics;
- caption metrics;
- robustness perturbations.

In [ ]:
reproducibility_record = pd.Series(
    {
        "module": (
            "Module 8 • Large Language Models"
        ),
        "lesson": (
            "Lesson 49 • Multimodal Large Language Models "
            "and Vision-Language Foundations"
        ),
        "synthetic images": len(
            synthetic_records
        ),
        "image size": IMAGE_SIZE,
        "VQA examples": len(
            vqa_examples
        ),
        "retrieval R@1": (
            retrieval_recall_at_k(
                similarity_matrix,
                1,
            )
        ),
        "retrieval MRR": (
            retrieval_mrr(
                similarity_matrix
            )
        ),
        "offline execution": True,
        "python": (
            platform.python_version()
        ),
    },
    name="Lesson 49 experiment",
)

reproducibility_record

# 45. Knowledge Check

1. What is a multimodal language model?
2. What does a vision encoder produce?
3. Why is a projection layer needed?
4. What is a shared image-text embedding space?
5. What is contrastive learning?
6. How does image-to-text retrieval work?
7. What is VQA?
8. How does captioning differ from classification?
9. What is multimodal fusion?
10. How do early and late fusion differ?
11. What is multimodal hallucination?
12. Why can OCR errors affect reasoning?
13. Why test visual perturbation robustness?
14. Why evaluate modality ablations?
15. What Arabic-specific issues affect multimodal NLP?

# 46. Exercises

## Exercise 1 — New Shapes
Add triangles to the synthetic image dataset.

## Exercise 2 — New Colors
Add yellow and purple.

## Exercise 3 — Contrastive Retrieval
Compare cosine retrieval before and after projection.

## Exercise 4 — VQA
Add size questions such as small versus large.

## Exercise 5 — Captioning
Add multi-object captions.

## Exercise 6 — Spatial Reasoning
Create two-object left/right relation questions.

## Exercise 7 — Perturbation Testing
Add blur, brightness changes, and occlusion.

## Exercise 8 — Neural Vision-Language Model
Replace the offline pipeline with a pretrained model.

## Exercise 9 — Arabic VQA
Create fully vocalized Arabic questions and answers.

## Exercise 10 — Multimodal Evaluation Card
Report retrieval, VQA, captioning, and robustness metrics.

## Challenge Exercises

1. Implement a small CNN image encoder.
2. Train a learned projection with contrastive loss.
3. Add two-image comparison questions.
4. Build an OCR-aware multimodal pipeline.
5. Create an English–Arabic image-text retrieval benchmark.

# 47. Summary and Next Lesson

In this lesson:

- multimodal AI and vision-language modeling were introduced;
- vision encoders, language models, and projection layers were connected;
- shared image-text embedding spaces and contrastive alignment were explained;
- synthetic image-text pairs were created;
- visual and textual feature encoders were implemented;
- cross-modal retrieval was evaluated with recall@k and MRR;
- visual question answering and caption generation were implemented offline;
- multimodal prompt templates, fusion, and modality ablation were introduced;
- robustness to visual perturbations was measured;
- OCR, spatial reasoning, hallucination, safety, multilingual, Arabic, and
  tashkeel considerations were integrated.

## Next Lesson

**Lesson 50: LLM Course Capstone — Building an End-to-End Intelligent NLP
Application** integrates prompting, RAG, evaluation, tool use, structured outputs,
multilingual handling, and reliability into a complete final project.

# References

- Radford, A. et al. *Learning Transferable Visual Models From Natural Language
  Supervision*.
- Alayrac, J.-B. et al. *Flamingo: a Visual Language Model for Few-Shot Learning*.
- Li, J. et al. work on BLIP and vision-language pretraining.
- Liu, H. et al. work on visual instruction tuning.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.